In [ ]:
# Code Cell 1: Kaggle Code 

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Imports and Environment Setup
import os
import sys
import warnings
import subprocess
import zipfile
import shutil
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor

# Wheel installation setup
WHEEL_DIR = '/kaggle/input/rdkit-mordred-wheels'

def debug_wheel_directory():
    """Debug what's actually in the wheel directory"""
    print(f"Debugging wheel directory: {WHEEL_DIR}")
    
    if os.path.exists(WHEEL_DIR):
        print("Wheel directory exists")
        try:
            files = os.listdir(WHEEL_DIR)
            print(f"Directory contents: {files}")
            
            wheel_files = [f for f in files if f.endswith('.whl')]
            print(f"Wheel files found: {wheel_files}")
            
            # Check file sizes and permissions
            for wheel in wheel_files:
                wheel_path = os.path.join(WHEEL_DIR, wheel)
                size = os.path.getsize(wheel_path)
                print(f"{wheel}: {size} bytes")
                
        except Exception as e:
            print(f"Error reading directory: {e}")
    else:
        print(f"Wheel directory does not exist: {WHEEL_DIR}")

def install_offline_verbose(package_name, import_name=None):
    """Install package with detailed debugging"""
    module_name = import_name or package_name
    
    print(f"Attempting to install {package_name} (import as {module_name})")
    
    # Check if already installed
    try:
        __import__(module_name)
        print(f"{module_name} already available")
        return True
    except ImportError:
        print(f"{module_name} not found, proceeding with installation")
    
    # Check wheel directory
    if not os.path.exists(WHEEL_DIR):
        print(f"Wheel directory not found: {WHEEL_DIR}")
        return False
    
    # Build and display command
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index", f"--find-links={WHEEL_DIR}", 
        "--verbose", "--no-cache-dir", package_name
    ]
    
    print(f"Command: {' '.join(cmd)}")
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        
        print(f"Return code: {result.returncode}")
        if result.stdout:
            print(f"STDOUT:\n{result.stdout}")
        if result.stderr:
            print(f"STDERR:\n{result.stderr}")
        
        if result.returncode == 0:
            # Test import
            try:
                __import__(module_name)
                print(f"{module_name} installed and imported successfully!")
                return True
            except ImportError as e:
                print(f"Installation succeeded but import failed: {e}")
                return False
        else:
            print(f"Installation failed with code {result.returncode}")
            return False
            
    except subprocess.TimeoutExpired:
        print("Installation timed out")
        return False
    except Exception as e:
        print(f"Installation error: {e}")
        return False

# Debug first
debug_wheel_directory()

# Try verbose installation
RDKIT_AVAILABLE = install_offline_verbose("rdkit")
MORDRED_AVAILABLE = install_offline_verbose("mordredcommunity", "mordred")

# Import packages if successful
if RDKIT_AVAILABLE:
    try:
        from rdkit import Chem, RDLogger
        from rdkit.Chem import Descriptors, rdMolDescriptors
        from rdkit.Chem import MACCSkeys
        from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect
        print("RDKit imports successful")
    except ImportError as e:
        print(f"RDKit import failed: {e}")
        RDKIT_AVAILABLE = False
        Chem, Descriptors, rdMolDescriptors = None, None, None
else:
    Chem, Descriptors, rdMolDescriptors = None, None, None

if MORDRED_AVAILABLE:
    try:
        from mordred import Calculator, descriptors
        print("Mordred imports successful")
    except ImportError as e:
        print(f"Mordred import failed: {e}")
        MORDRED_AVAILABLE = False
        Calculator, descriptors = None, None
else:
    Calculator, descriptors = None, None

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb
import xgboost as xgb

# Constants
TARGETS = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Final status report
print(f"Final Environment Setup Status:")
if RDKIT_AVAILABLE and Chem is not None:
    try:
        print(f"RDKit: Available (version {Chem.rdBase.rdkitVersion})")
    except:
        print("RDKit: Available (version unknown)")
else:
    print("RDKit: Not available - using basic features")

if MORDRED_AVAILABLE and Calculator is not None:
    try:
        calc = Calculator(descriptors, ignore_3D=True)
        print(f"Mordred: Available ({len(calc.descriptors)} descriptors)")
    except:
        print("Mordred: Available (basic)")
else:
    print("Mordred: Not available - using fallback")

print(f"Target variables: {TARGETS}")
print(f"Random state: {RANDOM_STATE}")

# Set strategy
if MORDRED_AVAILABLE:
    FEATURE_STRATEGY = "mordred"
    print("Strategy: Using Mordred descriptors (optimal)")
elif RDKIT_AVAILABLE:
    FEATURE_STRATEGY = "rdkit"
    print("Strategy: Using RDKit descriptors (good)")
else:
    FEATURE_STRATEGY = "basic"
    print("Strategy: Using basic chemical features (competitive)")

In [ ]:
# Constants
TARGETS = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
RANDOM_STATE = 42

# Official competition data
train_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')
test_df  = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

# Supplement Data 
supplement_1_tc = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv')
supplement_2 = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv')
supplement_3_tg = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv')
supplement_4_ffv = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv')

# Rename column for consistency
supplement_1_tc = supplement_1_tc.rename(columns={'TC_mean': 'Tc'})

print("Data loaded successfully:")
print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("supplement_1_tc", supplement_1_tc.shape)
print("supplement_2", supplement_2.shape)
print("supplement_3_tg", supplement_3_tg.shape)
print("supplement_4_ffv", supplement_4_ffv.shape)


In [ ]:
# Comprehensive EDA for All Polymer Prediction Datasets
print("COMPREHENSIVE EDA - NEURIPS POLYMER PREDICTION 2025")

# Define all your datasets for systematic analysis
datasets = {
    # Official Competition Data
    'train_df': train_df,
    'test_df': test_df,
    
    # Supplemental Competition Data  
    'supplement_1_tc': supplement_1_tc,
    'supplement_2': supplement_2,
    'supplement_3_tg': supplement_3_tg,
    'supplement_4_ffv': supplement_4_ffv,
}

print(f"ANALYZING {len(datasets)} DATASETS")

# 1. BASIC DATASET OVERVIEW
print("\nBASIC DATASET OVERVIEW")

overview_data = []
for name, df in datasets.items():
    overview_data.append({
        'Dataset': name,
        'Rows': f"{df.shape[0]:,}",
        'Columns': df.shape[1],
        'Memory_MB': f"{df.memory_usage(deep=True).sum() / 1024**2:.1f}",
        'Has_SMILES': 'SMILES' in df.columns,
        'Target_Columns': [col for col in TARGETS if col in df.columns]
    })

overview_df = pd.DataFrame(overview_data)
print(overview_df.to_string(index=False))

# 2. DETAILED ANALYSIS PER DATASET
print(f"\nDetailed Dataset Analysis")

for name, df in datasets.items():
    print(f"\nDATASET: {name}")
    
    # Basic Info
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Target Analysis (if targets present)
    target_cols = [col for col in TARGETS if col in df.columns]
    if target_cols:
        print(f"TARGET ANALYSIS:")
        for target in target_cols:
            data = df[target].dropna()
            if len(data) > 0:
                print(f"{target}:")
                print(f"  Samples: {len(data):,}")
                print(f"  Range: {data.min():.3f} to {data.max():.3f}")
                print(f"  Mean±Std: {data.mean():.3f}±{data.std():.3f}")
                print(f"  Missing: {df[target].isnull().sum():,} ({df[target].isnull().mean()*100:.1f}%)")

print("EDA Complete - Ready for Data Integration & Modeling!")


In [ ]:
def split_train_by_targets(train_df, targets, verbose=True):
    """
    Split the main training dataframe into separate dataframes per target.
    Each resulting dataframe contains SMILES + one target column with no missing values.
    """
    target_dataframes = {}
    
    if verbose:
        print("Splitting train_df by targets:")
        print(f"Original train_df: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
    
    for target in targets:
        if target in train_df.columns:
            # Create dataframe with SMILES + target column
            df_target = train_df[['SMILES', target]].copy()
            
            # Count missing values before filtering
            missing_count = df_target[target].isna().sum()
            
            # Remove rows where target is missing
            df_target = df_target.dropna(subset=[target])
            
            # Store in results
            target_dataframes[target] = df_target
            
            if verbose:
                print(f"{target}: {df_target.shape[0]:,} rows (dropped {missing_count:,} missing)")
        else:
            if verbose:
                print(f"{target}: not found in train_df columns")
            target_dataframes[target] = pd.DataFrame(columns=['SMILES', target])
    
    if verbose:
        print("Target splitting complete!")
    
    return target_dataframes

def combine_target_datasets(target_name, datasets_list, verbose=True):
    """
    Combine multiple datasets for a single target, deduplicate by SMILES.
    """
    if verbose:
        print(f"Combining datasets for {target_name}:")
    
    # Track original sizes
    valid_datasets = []
    
    for i, df in enumerate(datasets_list):
        if df is not None and len(df) > 0:
            # Ensure we have the right columns
            if 'SMILES' in df.columns and target_name in df.columns:
                # Keep only SMILES + target, drop missing targets
                df_clean = df[['SMILES', target_name]].dropna()
                valid_datasets.append(df_clean)
                if verbose:
                    print(f"Dataset {i+1}: {len(df_clean):,} valid rows")
            else:
                if verbose:
                    print(f"Dataset {i+1}: missing required columns, skipping")
    
    # Combine all datasets
    if valid_datasets:
        combined = pd.concat(valid_datasets, ignore_index=True)
        
        # Deduplicate by SMILES (keep first occurrence)
        combined_deduped = combined.drop_duplicates(subset=['SMILES'], keep='first').reset_index(drop=True)
        
        duplicates_removed = len(combined) - len(combined_deduped)
        
        if verbose:
            print(f"Combined: {len(combined):,} total rows")
            print(f"Deduplicated: {len(combined_deduped):,} unique SMILES")
            print(f"Removed: {duplicates_removed:,} duplicates")
            print(f"Final: {len(combined_deduped):,} rows × 2 columns")
        
        return combined_deduped
    else:
        if verbose:
            print(f"No valid datasets found for {target_name}")
        return pd.DataFrame(columns=['SMILES', target_name])

# Apply to your data
official_targets = split_train_by_targets(train_df, TARGETS, verbose=True)

# Combine datasets per target
print("\nCOMBINING ALL DATA SOURCES PER TARGET")

# Tg: official + supplement_3
combined_tg = combine_target_datasets('Tg', [
    official_targets['Tg'], 
    supplement_3_tg, 
], verbose=True)

# FFV: official + supplement_4
combined_ffv = combine_target_datasets('FFV', [
    official_targets['FFV'],
    supplement_4_ffv,
], verbose=True)

# Tc: official + supplement_1
combined_tc = combine_target_datasets('Tc', [
    official_targets['Tc'],
    supplement_1_tc,
], verbose=True)

# Density: official only
combined_density = combine_target_datasets('Density', [
    official_targets['Density'],
], verbose=True)

# Rg: official only
combined_rg = combine_target_datasets('Rg', [
    official_targets['Rg'],
], verbose=True)

# Store all combined datasets for easy access
combined_datasets = {
    'Tg': combined_tg,
    'FFV': combined_ffv, 
    'Tc': combined_tc,
    'Density': combined_density,
    'Rg': combined_rg
}

# Summary statistics
print(f"\nData Combination Complete")
total_samples = 0
for target, df in combined_datasets.items():
    print(f"{target:8}: {len(df):,} samples")
    total_samples += len(df)

print(f"{'Total':8}: {total_samples:,} samples across all targets")


In [ ]:
# Enhanced Mordred 2D + Fingerprints + TARGET-SPECIFIC Feature Extraction
# Combined Best Approaches: V49 for Tg, V48 for FFV, V52 for Tc/Density/Rg
# **ADJUSTED: Density now uses enhanced Tg sharing from Version 77**

from tqdm import tqdm
import multiprocessing
from joblib import Parallel, delayed
from rdkit.Chem import AllChem, Descriptors3D, Crippen

# Suppress warnings for cleaner output
if RDKIT_AVAILABLE:
    RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

print("ADJUSTED COMBINED BEST-PRACTICE FEATURE EXTRACTION PIPELINE")
print("Version 49 for Tg, Version 48 for FFV, Version 52 for Tc/Density/Rg")
print("DENSITY NOW ENHANCED WITH VERSION 77 Tg SHARING")
print("=" * 80)

# Get number of CPU cores for optimal parallelization
N_JOBS = min(4, multiprocessing.cpu_count())
print(f"Using {N_JOBS} parallel workers")

# Initialize Mordred calculator (do this once for efficiency)
if MORDRED_AVAILABLE:
    calc = Calculator(descriptors, ignore_3D=True)  # 2D descriptors only
    print(f"Mordred calculator initialized with {len(calc.descriptors)} 2D descriptors")
print("+ Morgan fingerprints (1024 bits)")
print("+ MACCS fingerprints (166 bits)")
print("+ Target-specific enhancements per best-performing version")
print("+ Enhanced Tg sharing for Density (Version 77 enhancement)")

# === VERSION 49 Tg-specific features (13 total features) ===
TG_FUNCTIONAL_GROUPS = {
    'amine_primary': '[N;H2;!$(NC=O)]',           # Primary amines (not amides)
    'amine_secondary': '[N;H1;!$(NC=O);!$(N=*)]', # Secondary amines (not amides/imines)  
    'lactam_5ring': '[#6]1[#6][#6][#6][#7]1[#6](=[#8])',      # 5-membered lactams
    'lactam_6ring': '[#6]1[#6][#6][#6][#6][#7]1[#6](=[#8])',  # 6-membered lactams
    'oxoheterocycle': '[#6](=[#8])-[#6]1:[#6]:[#6]:[#6]:[#6]:[#7]:1', # Oxo-heteroaromatics
    'aromatic_amine': '[c][N;H2,H1]',             # Aromatic amines
    'hydroxyl_aromatic': '[c][OH]',               # Aromatic hydroxyl
    'ether_aromatic': '[c][O][c]'                 # Aromatic ethers
}

def extract_tg_specific_features_v49(mol):
    """Extract Tg-specific molecular features from Version 49 (best for Tg)"""
    if mol is None:
        return {
            'rotatable_bonds': 0,
            'balaban_j': 0.0,
            'max_estate_idx': 0.0,
            'min_abs_estate': 0.0,
            'aromatic_ratio': 0.0,
            **{name: 0 for name in TG_FUNCTIONAL_GROUPS.keys()}
        }
    
    try:
        features = {}
        
        # 1. Rotational descriptors (most critical for Tg)
        features['rotatable_bonds'] = rdMolDescriptors.CalcNumRotatableBonds(mol)
        
        # 2. Rigidity indicators  
        features['balaban_j'] = Descriptors.BalabanJ(mol)
        features['max_estate_idx'] = Descriptors.MaxEStateIndex(mol)
        features['min_abs_estate'] = Descriptors.MinAbsEStateIndex(mol)
        
        # 3. Aromaticity (backbone rigidity proxy)
        num_aromatic = sum(1 for atom in mol.GetAtoms() if atom.GetIsAromatic())
        total_atoms = mol.GetNumAtoms()
        features['aromatic_ratio'] = num_aromatic / total_atoms if total_atoms > 0 else 0.0
        
        # 4. Functional group counts (chemical specificity for Tg)
        for group_name, smarts in TG_FUNCTIONAL_GROUPS.items():
            pattern = Chem.MolFromSmarts(smarts)
            if pattern:
                matches = len(mol.GetSubstructMatches(pattern))
                features[group_name] = matches
            else:
                features[group_name] = 0
                
        return features
        
    except Exception:
        # Return zeros if calculation fails
        return {
            'rotatable_bonds': 0,
            'balaban_j': 0.0,
            'max_estate_idx': 0.0,
            'min_abs_estate': 0.0,
            'aromatic_ratio': 0.0,
            **{name: 0 for name in TG_FUNCTIONAL_GROUPS.keys()}
        }

# === ENHANCED Tg features for sharing (VERSION 77 ENHANCEMENT) ===
def extract_tg_specific_features_v49_enhanced(mol):
    """Extract ENHANCED Tg-specific molecular features (20 features total) - FROM VERSION 77"""
    if mol is None:
        return {
            # Original 13 features
            'rotatable_bonds': 0, 'balaban_j': 0.0, 'max_estate_idx': 0.0,
            'min_abs_estate': 0.0, 'aromatic_ratio': 0.0,
            **{name: 0 for name in TG_FUNCTIONAL_GROUPS.keys()},
            # NEW 7 additional features
            'mol_logp': 0.0, 'exact_mol_wt': 0.0, 'tpsa': 0.0,
            'asphericity_3d': 0.0, 'eccentricity_3d': 0.0, 
            'spherocity_3d': 0.0, 'radius_gyration_3d': 0.0
        }
    
    try:
        features = {}
        
        # === ORIGINAL 13 Tg features ===
        features['rotatable_bonds'] = rdMolDescriptors.CalcNumRotatableBonds(mol)
        features['balaban_j'] = Descriptors.BalabanJ(mol)
        features['max_estate_idx'] = Descriptors.MaxEStateIndex(mol)
        features['min_abs_estate'] = Descriptors.MinAbsEStateIndex(mol)
        
        # Aromaticity ratio
        num_aromatic = sum(1 for atom in mol.GetAtoms() if atom.GetIsAromatic())
        total_atoms = mol.GetNumAtoms()
        features['aromatic_ratio'] = num_aromatic / total_atoms if total_atoms > 0 else 0.0
        
        # Functional groups
        for group_name, smarts in TG_FUNCTIONAL_GROUPS.items():
            pattern = Chem.MolFromSmarts(smarts)
            if pattern:
                matches = len(mol.GetSubstructMatches(pattern))
                features[group_name] = matches
            else:
                features[group_name] = 0
        
        # === NEW 7 additional Tg features FROM VERSION 77 ===
        # 2D descriptors relevant to glass transition
        features['mol_logp'] = Crippen.MolLogP(mol)  # Lipophilicity affects Tg
        features['exact_mol_wt'] = Descriptors.ExactMolWt(mol)  # Molecular weight affects Tg
        features['tpsa'] = Descriptors.TPSA(mol)  # Polar surface area affects chain packing
        
        # 3D descriptors (generate conformer first)
        try:
            mol_h = Chem.AddHs(mol)
            if AllChem.EmbedMolecule(mol_h, AllChem.ETKDG()) == 0:
                AllChem.MMFFOptimizeMolecule(mol_h)
                
                # 3D shape descriptors critical for glass transition
                features['asphericity_3d'] = Descriptors3D.Asphericity(mol_h)
                features['eccentricity_3d'] = Descriptors3D.Eccentricity(mol_h) 
                features['spherocity_3d'] = Descriptors3D.SpherocityIndex(mol_h)
                features['radius_gyration_3d'] = Descriptors3D.RadiusOfGyration(mol_h)
            else:
                # Fallback if 3D generation fails
                features.update({
                    'asphericity_3d': 0.0, 'eccentricity_3d': 0.0,
                    'spherocity_3d': 0.0, 'radius_gyration_3d': 0.0
                })
        except:
            features.update({
                'asphericity_3d': 0.0, 'eccentricity_3d': 0.0, 
                'spherocity_3d': 0.0, 'radius_gyration_3d': 0.0
            })
        
        return features
        
    except Exception:
        # Return zeros if calculation fails
        return {
            'rotatable_bonds': 0, 'balaban_j': 0.0, 'max_estate_idx': 0.0,
            'min_abs_estate': 0.0, 'aromatic_ratio': 0.0,
            **{name: 0 for name in TG_FUNCTIONAL_GROUPS.keys()},
            'mol_logp': 0.0, 'exact_mol_wt': 0.0, 'tpsa': 0.0,
            'asphericity_3d': 0.0, 'eccentricity_3d': 0.0,
            'spherocity_3d': 0.0, 'radius_gyration_3d': 0.0
        }

# === VERSION 52 Target-specific features for Tc, Density, Rg ===

def extract_tg_features_v52_for_sharing(mol):
    """Extract 13 Tg features for sharing with Tc and Density (Version 52 approach) - ORIGINAL"""
    return extract_tg_specific_features_v49(mol)  # Same 13 features

def extract_tg_features_v52_for_sharing_enhanced(mol):
    """Extract 20 enhanced Tg features for sharing with Density (VERSION 77 ENHANCEMENT)"""
    return extract_tg_specific_features_v49_enhanced(mol)  # Enhanced 20 features

def extract_tc_features(mol):
    """Extract Tc-specific features (Version 52)"""
    if mol is None:
        return {
            'molecular_weight': 0.0,
            'num_rings': 0,
            'pi_electrons': 0,
            'chain_flexibility': 0.0
        }
    
    try:
        features = {}
        
        features['molecular_weight'] = Descriptors.MolWt(mol)
        features['num_rings'] = Descriptors.RingCount(mol)
        features['pi_electrons'] = Descriptors.NumAromaticRings(mol) * 6  # Rough estimate
        features['chain_flexibility'] = rdMolDescriptors.CalcNumRotatableBonds(mol) / max(mol.GetNumAtoms(), 1)
        
        return features
        
    except Exception:
        return {
            'molecular_weight': 0.0,
            'num_rings': 0,
            'pi_electrons': 0,
            'chain_flexibility': 0.0
        }

def extract_density_features(mol):
    """Extract Density-specific features (Version 52)"""
    if mol is None:
        return {
            'heavy_atom_count': 0,
            'fraction_sp3': 0.0,
            'molecular_compactness': 0.0,
            'branching_index': 0.0
        }
    
    try:
        features = {}
        
        features['heavy_atom_count'] = Descriptors.HeavyAtomCount(mol)
        features['fraction_sp3'] = Descriptors.FractionCSP3(mol)
        features['molecular_compactness'] = Descriptors.HeavyAtomCount(mol) / max(Descriptors.MolWt(mol), 1)
        features['branching_index'] = Descriptors.BertzCT(mol) / max(mol.GetNumAtoms(), 1)
        
        return features
        
    except Exception:
        return {
            'heavy_atom_count': 0,
            'fraction_sp3': 0.0,
            'molecular_compactness': 0.0,
            'branching_index': 0.0
        }

def extract_rg_features(mol):
    """Extract Rg-specific features (Version 52 - original 4 features)"""
    if mol is None:
        return {
            'num_aromatic_rings': 0,
            'chain_length_proxy': 0,
            'molecular_eccentricity': 0.0,
            'rigidity_index': 0.0
        }
    
    try:
        features = {}
        
        # Original 4 Rg features
        features['num_aromatic_rings'] = Descriptors.NumAromaticRings(mol)
        features['chain_length_proxy'] = mol.GetNumAtoms()
        features['molecular_eccentricity'] = Descriptors.Eccentricity(mol)
        
        # Rigidity vs flexibility
        rigid_bonds = Descriptors.NumAromaticRings(mol) * 6
        total_bonds = mol.GetNumBonds()
        features['rigidity_index'] = rigid_bonds / max(total_bonds, 1)
        
        return features
        
    except Exception:
        return {
            'num_aromatic_rings': 0,
            'chain_length_proxy': 0,
            'molecular_eccentricity': 0.0,
            'rigidity_index': 0.0
        }

def extract_fingerprints_batch(mol_batch):
    """Extract fingerprints for a batch of molecules"""
    morgan_features = []
    maccs_features = []
    
    for mol in mol_batch:
        if mol is not None:
            try:
                # Morgan fingerprint (1024 bits, radius 2)
                morgan_fp = GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
                morgan_feat = list(morgan_fp)
                
                # MACCS fingerprint - Drop bit 0 to get 166 bits
                maccs_fp = MACCSkeys.GenMACCSKeys(mol)
                maccs_feat = list(maccs_fp)[1:]  # Drop first bit (index 0)
                
                morgan_features.append(morgan_feat)
                maccs_features.append(maccs_feat)
            except:
                morgan_features.append([0] * 1024)
                maccs_features.append([0] * 166)
        else:
            # Failed molecules get zero fingerprints
            morgan_features.append([0] * 1024)
            maccs_features.append([0] * 166)
    
    return morgan_features, maccs_features

def extract_base_features(smiles_list, dataset_name):
    """Extract base Mordred + Morgan + MACCS features (common to all targets)"""
    print(f"   Processing {len(smiles_list):,} molecules for {dataset_name}")
    
    # Step 1: Parse SMILES to molecules
    print("   Step 1/3: Parsing SMILES strings...")
    molecules = []
    failed_parsing = 0
    
    for smiles in tqdm(smiles_list, desc="Parsing SMILES"):
        try:
            clean_smiles = str(smiles).strip().replace(" ", "")
            mol = Chem.MolFromSmiles(clean_smiles)
            
            if mol is None:
                clean_smiles = clean_smiles.replace("[", "").replace("]", "")
                mol = Chem.MolFromSmiles(clean_smiles)
            
            if mol is None:
                failed_parsing += 1
                
            molecules.append(mol)
        except:
            molecules.append(None)
            failed_parsing += 1
    
    success_rate = ((len(molecules) - failed_parsing) / len(molecules)) * 100
    print(f"   Molecule parsing: {len(smiles_list) - failed_parsing:,} successful ({success_rate:.1f}%)")
    
    # Step 2: Calculate Mordred descriptors
    print(f"   Step 2/3: Computing Mordred 2D descriptors...")
    if MORDRED_AVAILABLE:
        desc_df = calc.pandas(molecules)
        
        # Clean Mordred descriptors
        desc_df.columns = desc_df.columns.map(str)
        desc_df = desc_df.select_dtypes(include=[np.number]).copy()
        desc_df.replace([np.inf, -np.inf], np.nan, inplace=True)
        desc_df = desc_df.fillna(0.0)
        
        # Remove constant columns (no variance = no information)
        initial_features = desc_df.shape[1]
        feature_variances = desc_df.var()
        constant_mask = feature_variances > 1e-8  # Very small threshold
        desc_df = desc_df.loc[:, constant_mask]
        
        removed_features = initial_features - desc_df.shape[1]
        print(f"   Removed {removed_features:,} constant Mordred features")
        print(f"   Final Mordred features: {desc_df.shape[1]:,}")
    else:
        # Fallback: create basic features
        print("   Mordred not available, creating basic features...")
        basic_features = []
        for mol in molecules:
            if mol is not None:
                basic_features.append({
                    'mol_weight': Descriptors.MolWt(mol),
                    'num_atoms': mol.GetNumAtoms(),
                    'num_bonds': mol.GetNumBonds(),
                })
            else:
                basic_features.append({'mol_weight': 0, 'num_atoms': 0, 'num_bonds': 0})
        desc_df = pd.DataFrame(basic_features)
        print(f"   Basic features: {desc_df.shape[1]:,}")
    
    # Step 3: Extract fingerprints
    print(f"   Step 3/3: Computing fingerprints...")
    
    # Process fingerprints in batches for efficiency
    batch_size = max(100, len(molecules) // N_JOBS) if len(molecules) > 1000 else len(molecules)
    morgan_features = []
    maccs_features = []
    
    for i in tqdm(range(0, len(molecules), batch_size), desc="Fingerprint batches"):
        batch = molecules[i:i + batch_size]
        morgan_batch, maccs_batch = extract_fingerprints_batch(batch)
        morgan_features.extend(morgan_batch)
        maccs_features.extend(maccs_batch)
    
    # Create fingerprint dataframes
    morgan_df = pd.DataFrame(morgan_features, columns=[f"Morgan_{i}" for i in range(1024)])
    maccs_df = pd.DataFrame(maccs_features, columns=[f"MACCS_{i}" for i in range(166)])
    
    print(f"   Morgan features: 1,024")
    print(f"   MACCS features: 166")
    
    return desc_df, morgan_df, maccs_df, molecules

print("Feature extraction functions loaded successfully!")

# === Target-specific feature extraction functions ===

def extract_features_tg_v49(smiles_list, dataset_name):
    """Extract features for Tg using Version 49 approach (best for Tg)"""
    print(f"Extracting Tg features using VERSION 49 approach for {dataset_name}")
    
    # Get base features
    desc_df, morgan_df, maccs_df, molecules = extract_base_features(smiles_list, dataset_name)
    
    # Extract Tg-specific features
    print(f"   Extracting 13 Tg-specific features...")
    tg_features_list = []
    for mol in tqdm(molecules, desc="Tg-specific features"):
        tg_feat = extract_tg_specific_features_v49(mol)
        tg_features_list.append(tg_feat)
    
    tg_df = pd.DataFrame(tg_features_list)
    print(f"   Added {tg_df.shape[1]} Tg-specific features")
    
    # Combine all features
    combined_df = pd.concat([desc_df, tg_df, morgan_df, maccs_df], axis=1)
    
    print(f"Total Tg features: {combined_df.shape[1]:,}")
    print(f"Breakdown: Mordred({desc_df.shape[1]}) + Tg-specific({tg_df.shape[1]}) + Morgan(1024) + MACCS(166)")
    
    return combined_df.values, list(combined_df.columns)

def extract_features_ffv_v48(smiles_list, dataset_name):
    """Extract features for FFV using Version 48 approach (best for FFV)"""
    print(f"Extracting FFV features using VERSION 48 approach for {dataset_name}")
    
    # Get base features only (no target-specific features for FFV in V48)
    desc_df, morgan_df, maccs_df, molecules = extract_base_features(smiles_list, dataset_name)
    
    # Combine base features only (Version 48 had no FFV-specific features)
    combined_df = pd.concat([desc_df, morgan_df, maccs_df], axis=1)
    
    print(f"Total FFV features: {combined_df.shape[1]:,}")
    print(f"   Breakdown: Mordred({desc_df.shape[1]}) + Morgan(1024) + MACCS(166)")
    
    return combined_df.values, list(combined_df.columns)

def extract_features_tc_v52(smiles_list, dataset_name):
    """Extract features for Tc using Version 52 approach (best for Tc)"""
    print(f"Extracting Tc features using VERSION 52 approach for {dataset_name}")
    
    # Get base features
    desc_df, morgan_df, maccs_df, molecules = extract_base_features(smiles_list, dataset_name)
    
    # Extract Tc-specific + shared Tg features (ORIGINAL 13 features)
    print(f"   Extracting Tc-specific + shared Tg features...")
    target_features_list = []
    for mol in tqdm(molecules, desc="Tc + Tg features"):
        tc_feat = extract_tc_features(mol)
        tg_feat = extract_tg_features_v52_for_sharing(mol)  # ORIGINAL 13 shared Tg features
        combined_feat = {**tc_feat, **{f"tg_{k}": v for k, v in tg_feat.items()}}
        target_features_list.append(combined_feat)
    
    target_df = pd.DataFrame(target_features_list)
    print(f"Added {target_df.shape[1]} target-specific features (4 Tc + 13 shared Tg)")
    
    # Combine all features
    combined_df = pd.concat([desc_df, target_df, morgan_df, maccs_df], axis=1)
    
    print(f"Total Tc features: {combined_df.shape[1]:,}")
    print(f"Breakdown: Mordred({desc_df.shape[1]}) + Tc+Tg({target_df.shape[1]}) + Morgan(1024) + MACCS(166)")
    
    return combined_df.values, list(combined_df.columns)

def extract_features_density_v52_enhanced(smiles_list, dataset_name):
    """ENHANCED Extract features for Density using VERSION 77 Tg sharing approach"""
    print(f"Extracting ENHANCED Density features using VERSION 77 Tg sharing for {dataset_name}")
    
    # Get base features
    desc_df, morgan_df, maccs_df, molecules = extract_base_features(smiles_list, dataset_name)
    
    # Extract Density-specific + shared ENHANCED Tg features (VERSION 77 ENHANCEMENT)
    print(f"   Extracting Density-specific + shared ENHANCED Tg features...")
    target_features_list = []
    for mol in tqdm(molecules, desc="Density + Enhanced Tg features"):
        density_feat = extract_density_features(mol)  # Keep original 4 features
        tg_feat = extract_tg_features_v52_for_sharing_enhanced(mol)  # 20 enhanced shared features
        combined_feat = {**density_feat, **{f"tg_{k}": v for k, v in tg_feat.items()}}
        target_features_list.append(combined_feat)
    
    target_df = pd.DataFrame(target_features_list)
    print(f"Added {target_df.shape[1]} target-specific features (4 Density + 20 shared enhanced Tg)")
    
    # Combine all features
    combined_df = pd.concat([desc_df, target_df, morgan_df, maccs_df], axis=1)
    
    print(f"Total ENHANCED Density features: {combined_df.shape[1]:,}")
    print(f"Breakdown: Mordred({desc_df.shape[1]}) + Density+Tg({target_df.shape[1]}) + Morgan(1024) + MACCS(166)")
    
    return combined_df.values, list(combined_df.columns)

def extract_features_rg_v52(smiles_list, dataset_name):
    """Extract features for Rg using Version 52 approach (best for Rg)"""
    print(f"Extracting Rg features using VERSION 52 approach for {dataset_name}")
    
    # Get base features
    desc_df, morgan_df, maccs_df, molecules = extract_base_features(smiles_list, dataset_name)
    
    # Extract Rg-specific features (isolated, no sharing)
    print(f"   Extracting 4 Rg-specific features (isolated)...")
    rg_features_list = []
    for mol in tqdm(molecules, desc="Rg-specific features"):
        rg_feat = extract_rg_features(mol)
        rg_features_list.append(rg_feat)
    
    rg_df = pd.DataFrame(rg_features_list)
    print(f"Added {rg_df.shape[1]} Rg-specific features")
    
    # Combine all features
    combined_df = pd.concat([desc_df, rg_df, morgan_df, maccs_df], axis=1)
    
    print(f"Total Rg features: {combined_df.shape[1]:,}")
    print(f"Breakdown: Mordred({desc_df.shape[1]}) + Rg-specific({rg_df.shape[1]}) + Morgan(1024) + MACCS(166)")
    
    return combined_df.values, list(combined_df.columns)

def extract_features_by_target(smiles_list, target, dataset_name):
    """ADJUSTED dispatcher for target-specific feature extraction - DENSITY ENHANCED"""
    
    if target == 'Tg':
        return extract_features_tg_v49(smiles_list, dataset_name)
    elif target == 'FFV':
        return extract_features_ffv_v48(smiles_list, dataset_name)
    elif target == 'Tc':
        return extract_features_tc_v52(smiles_list, dataset_name)
    elif target == 'Density':
        return extract_features_density_v52_enhanced(smiles_list, dataset_name)  # 🔥 ENHANCED
    elif target == 'Rg':
        return extract_features_rg_v52(smiles_list, dataset_name)
    else:
        raise ValueError(f"Unknown target: {target}")

def process_single_target_features(target_dataset_pair):
    """Process features for a single target - safe for parallel execution"""
    target, dataset = target_dataset_pair
    
    print(f"\nProcessing {target} with best-practice feature extraction...")
    
    # Extract features using target-specific best approach
    features, feature_names = extract_features_by_target(
        dataset['SMILES'].tolist(),
        target,
        f"{target}_combined"
    )
    
    return target, features, feature_names, dataset[target].values

def create_stratified_split(data_tuple):
    """Create stratified train/validation split - safe for parallel execution"""
    X, y, target_name = data_tuple
    
    try:
        # Create quantile bins for stratification
        n_bins = min(5, len(np.unique(y)))  # Max 5 bins, fewer if not enough unique values
        if n_bins >= 3:
            y_binned = pd.qcut(y, q=n_bins, duplicates='drop', labels=False)
            stratify = y_binned
        else:
            stratify = None
    except:
        stratify = None
    
    # 80/20 split with reproducible random state
    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=stratify
    )
    
    return target_name, {
        'X_train': X_train,
        'X_val': X_val, 
        'y_train': y_train,
        'y_val': y_val
    }

print("ADJUSTED Target-specific feature extraction functions ready!")

def main_feature_extraction_pipeline():
    """
    ADJUSTED Main pipeline for extracting features - DENSITY ENHANCED
    """
    
    print(f"\nADJUSTED COMBINED BEST-PRACTICE FEATURE EXTRACTION PIPELINE")
    print("-" * 80)
    
    # Store per-target feature matrices and schemas
    target_features = {}
    target_feature_schemas = {}
    target_test_features = {}
    
    # Extract test features for each target separately
    print(f"\nProcessing test set with adjusted target-specific best practices...")
    for target in TARGETS:
        print(f"\nExtracting test features for {target}:")
        test_features_target, test_feature_names_target = extract_features_by_target(
            test_df['SMILES'].tolist(), 
            target,
            f"test_set_{target}"
        )
        
        # Store per-target test features and schemas
        target_test_features[target] = test_features_target
        target_feature_schemas[target] = {
            'feature_names': test_feature_names_target,
            'n_features': len(test_feature_names_target)
        }
        
        print(f"{target} test features: {test_features_target.shape}")
    
    # Process training datasets
    target_datasets = [(target, combined_datasets[target]) 
                      for target in TARGETS if target in combined_datasets]
    
    # Process targets sequentially for stability
    print(f"\nProcessing targets sequentially...")
    for target in TARGETS:
        if target in combined_datasets:
            dataset = combined_datasets[target]
            
            # Extract features using best approach for this target
            features, feature_names = extract_features_by_target(
                dataset['SMILES'].tolist(),
                target,
                f"{target}_training"
            )
            
            # Align with test features for same target
            expected_features = target_feature_schemas[target]['feature_names']
            if set(feature_names) != set(expected_features):
                print(f"Feature alignment needed for {target}")
                aligned_features = np.zeros((features.shape[0], len(expected_features)))
                
                for i, expected_feature in enumerate(expected_features):
                    if expected_feature in feature_names:
                        source_idx = feature_names.index(expected_feature)
                        aligned_features[:, i] = features[:, source_idx]
                
                features = aligned_features
            
            target_features[target] = {
                'X': features,
                'y': dataset[target].values,
                'smiles': dataset['SMILES'].values
            }
            
            print(f"{target}: {features.shape[0]:,} samples × {features.shape[1]:,} features")
    
    # Parallel Train/Validation Splitting
    print(f"\nPARALLEL TRAIN/VALIDATION SPLITTING")
    print("=" * 70)
    
    split_tasks = [(target_features[target]['X'], target_features[target]['y'], target) 
                   for target in TARGETS if target in target_features]
    
    if len(split_tasks) > 1:
        print(f"Creating splits for {len(split_tasks)} targets in parallel...")
        
        split_results = Parallel(n_jobs=min(len(split_tasks), N_JOBS), backend='loky')(
            delayed(create_stratified_split)(task) for task in split_tasks
        )
    else:
        split_results = [create_stratified_split(split_tasks[0])] if split_tasks else []
    
    # Store split results
    target_splits = {}
    for target_name, split_data in split_results:
        target_splits[target_name] = split_data
        print(f"{target_name:8}: Train {split_data['X_train'].shape[0]:,} | "
              f"Val {split_data['X_val'].shape[0]:,} samples")
    
    # Final Summary
    print(f"\nADJUSTED COMBINED BEST-PRACTICE FEATURE EXTRACTION COMPLETE!")
    print("=" * 70)
    
    print(f"ADJUSTED FEATURE EXTRACTION SUMMARY:")
    for target in TARGETS:
        if target in target_feature_schemas:
            n_features = target_feature_schemas[target]['n_features']
            
            if target == 'Tg':
                approach = "Version 49 (13 Tg-specific features)"
            elif target == 'FFV':
                approach = "Version 48 (Mordred + fingerprints only)"
            elif target == 'Tc':
                approach = "Version 52 (4 Tc + 13 shared Tg features)"  
            elif target == 'Density':
                approach = "ENHANCED Version 52 (4 Density + 20 shared enhanced Tg features)"
            elif target == 'Rg':
                approach = "Version 52 (4 isolated Rg features)"
            else:
                approach = "Standard approach"
                
            print(f"{target:8}: {n_features:,} features - {approach}")
    
    # Performance metrics
    total_train_samples = sum(target_splits[t]['X_train'].shape[0] for t in target_splits.keys())
    total_val_samples = sum(target_splits[t]['X_val'].shape[0] for t in target_splits.keys())
    
    print(f"\nDATASET SUMMARY:")
    print(f"Training samples: {total_train_samples:,}")
    print(f"Validation samples: {total_val_samples:,}")
    print(f"Test samples: {len(test_df):,} (processed per target)")
    
    print(f"\nAvailable variables for model training:")
    print(f"• target_splits[target] - training/validation data per target")
    print(f"• target_test_features[target] - test set features per target") 
    print(f"• target_feature_schemas[target] - feature names and schema per target")
    
    print(f"\nREADY FOR MODEL TRAINING WITH ADJUSTED BEST-PRACTICE FEATURE EXTRACTION!")
    print(f"KEY ADJUSTMENT: Density now uses 20 enhanced Tg features (instead of 13)")
    print(f"   This includes 4 3D descriptors that achieved 0.01 MAE in Version 77!")
    
    return target_splits, target_test_features, target_feature_schemas

# Run the adjusted feature extraction pipeline
target_splits, target_test_features, target_feature_schemas = main_feature_extraction_pipeline()


In [ ]:
# HYBRID TARGET-SPECIFIC MODEL TRAINING WITH VERSION-BASED ENSEMBLE SELECTION
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor  
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge, ElasticNet, Lasso, BayesianRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_predict, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

print("HYBRID TARGET-SPECIFIC MODEL TRAINING WITH VERSION-BASED OPTIMIZATION")
print("=" * 80)

def get_target_model_config(target_name):
    if target_name == 'Tg':
        return {
            'LightGBM': lambda: LGBMRegressor(n_estimators=1000, learning_rate=0.1, max_depth=8, random_state=RANDOM_STATE, verbosity=-1),
            'XGBoost' : lambda: XGBRegressor(n_estimators=1000, learning_rate=0.1, max_depth=8, random_state=RANDOM_STATE, verbosity=0),
            'CatBoost': lambda: CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=8, random_seed=RANDOM_STATE, verbose=False)
        }
    elif target_name == 'FFV':
        return {
            'LightGBM': lambda: LGBMRegressor(n_estimators=1000, learning_rate=0.1, max_depth=8, random_state=RANDOM_STATE, verbosity=-1),
            'XGBoost' : lambda: XGBRegressor(n_estimators=1000, learning_rate=0.1, max_depth=8, random_state=RANDOM_STATE, verbosity=0),
            'CatBoost': lambda: CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=8, random_seed=RANDOM_STATE, verbose=False)
        }
    else:
        return {
            'LightGBM': lambda: LGBMRegressor(n_estimators=1000, learning_rate=0.08, max_depth=6, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5, min_child_samples=20, random_state=RANDOM_STATE, verbosity=-1, n_jobs=1),
            'XGBoost' : lambda: XGBRegressor(n_estimators=1000, learning_rate=0.08, max_depth=6, subsample=0.8, colsample_bytree=0.8, min_child_weight=3, reg_alpha=0.01, random_state=RANDOM_STATE, verbosity=0, n_jobs=1),
            'CatBoost': lambda: CatBoostRegressor(iterations=1000, learning_rate=0.08, depth=6, l2_leaf_reg=5, subsample=0.8, random_seed=RANDOM_STATE, verbose=False)
        }

# TARGET-SPECIFIC META MODEL CONFIGURATIONS
def get_meta_models_for_target(target_name, y_train_size):
    """Get target-specific meta models based on version requirements"""
    
    if target_name == 'Tg':
        # Version 49 approach - 14 meta models (SVR_poly removed)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }
    
    elif target_name == 'FFV':
        # Version 48 approach - 15 meta models (includes SVR_poly)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }
    
    else:  # Tc, Density, Rg
        # Version 52 approach - 14 meta models (SVR_poly removed)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }

def compute_data_driven_clipping_ranges(combined_datasets, targets, margin_pct=0.05):
    print("Computing data-driven clipping ranges:")
    PROPERTY_RANGES = {}
    for target in targets:
        if target in combined_datasets:
            y_values = combined_datasets[target][target].values
            y_min, y_max = y_values.min(), y_values.max()
            y_range = y_max - y_min
            margin = y_range * margin_pct
            lower, upper = y_min - margin, y_max + margin
            PROPERTY_RANGES[target] = (lower, upper)
            print(f"   {target:8}: Data range [{y_min:.3f}, {y_max:.3f}] → Clip range [{lower:.3f}, {upper:.3f}](±{margin_pct*100:.0f}%)")
        else:
            PROPERTY_RANGES[target] = (None, None)
            print(f"   {target:8}: No clipping (no data found)")
    print("Data-driven clipping ranges computed")
    return PROPERTY_RANGES

PROPERTY_RANGES = compute_data_driven_clipping_ranges(combined_datasets, TARGETS)

def create_target_specific_mlp(target_name, random_state=RANDOM_STATE):
    if target_name in ['Tg', 'Tc', 'Rg']:
        return make_pipeline(
            RobustScaler(), 
            MLPRegressor(
                hidden_layer_sizes=(100, 50, 25),    # Deeper 3-layer architecture
                activation='relu',                    # Explicit activation function
                alpha=0.005,                         # Better L2 regularization (was 0.01)
                solver='adam',                       # Explicit solver specification
                learning_rate='adaptive',            # Keep adaptive learning
                learning_rate_init=0.0005,          # Lower initial LR (was 0.0007)
                max_iter=1500,                       # Keep existing max_iter
                early_stopping=True,
                validation_fraction=0.15,            # Smaller validation split (was 0.2)
                n_iter_no_change=30,                 # More patience (was 25)
                beta_1=0.9,                         # Adam momentum parameter
                beta_2=0.999,                       # Adam momentum parameter
                epsilon=1e-8,                       # Adam numerical stability
                random_state=random_state
            )
        )
    else:  # FFV, Density
        return make_pipeline(
            RobustScaler(), 
            MLPRegressor(
                hidden_layer_sizes=(64, 32),        # Larger architecture (was 32,16)
                activation='tanh',                   # Different activation for variety
                alpha=0.005,                        # Better regularization (was 0.01)
                solver='adam',                      # Explicit solver
                learning_rate='adaptive',           # Add adaptive learning
                learning_rate_init=0.001,          # Reasonable learning rate
                max_iter=800,                       # More iterations (was 500)
                early_stopping=True,
                validation_fraction=0.15,           # Smaller validation split
                n_iter_no_change=20,               # Patience for early stopping
                beta_1=0.9,                        # Adam parameters
                beta_2=0.999,
                epsilon=1e-8,
                random_state=random_state
            )
        )


def train_single_model_with_early_stopping(X_train, y_train, X_val, y_val, model_func, model_name, target_name):
    model = model_func()
    use_enhanced = target_name in ['Tc', 'Density', 'Rg']
    try:
        if model_name == 'LightGBM' and use_enhanced:
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
        elif model_name == 'XGBoost' and use_enhanced:
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
        elif model_name == 'CatBoost' and use_enhanced:
            model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
        elif model_name in ['LightGBM', 'XGBoost'] and not use_enhanced:
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=100, verbose=False)
        else:
            model.fit(X_train, y_train)
    except Exception:
        model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)
    X_full = np.vstack([X_train, X_val])
    y_full = np.concatenate([y_train, y_val])
    oof_pred = cross_val_predict(model, X_full, y_full, cv=5, method='predict')
    return {'model': model, 'val_pred': val_pred, 'val_mae': val_mae, 'oof_pred': oof_pred, 'name': model_name}

def train_target_models_parallel(target_name, X_train, y_train, X_val, y_val):
    models_config = get_target_model_config(target_name)
    results = Parallel(n_jobs=3, verbose=0)(
        delayed(train_single_model_with_early_stopping)(
            X_train, y_train, X_val, y_val, model_func, model_name, target_name
        ) for model_name, model_func in models_config.items()
    )
    for r in results:
        print(f"   {r['name']:10}: Validation MAE = {r['val_mae']:.6f}")
    return results

def create_weighted_ensemble(results):
    maes = np.array([r['val_mae'] for r in results])
    weights = 1.0 / maes
    weights = weights / weights.sum()
    ensemble_pred = np.zeros_like(results[0]['val_pred'])
    for weight, result in zip(weights, results): 
        ensemble_pred += weight * result['val_pred']
    return ensemble_pred, weights

def create_stacked_ensemble(results, y_val):
    X_meta_val = np.column_stack([r['val_pred'] for r in results])
    meta_model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    meta_model.fit(X_meta_val, y_val)
    stacked_pred = meta_model.predict(X_meta_val)
    return stacked_pred, meta_model

def create_meta_learned_ensemble_safe(results, X_train, y_train, X_val, y_val, target_name):
    """Create meta-learned ensemble with target-specific meta models from different versions"""
    print(f"   Safe meta-learning for {target_name} using version-specific approach...")
    
    # Step 1: Build training meta-features from OOF predictions on training data ONLY
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    meta_features_train = np.zeros((X_train.shape[0], len(results)))
    
    for i, result in enumerate(results):
        model_func = [func for name, func in get_target_model_config(target_name).items() if name == result['name']][0]
        fresh_model = model_func()
        oof_pred = cross_val_predict(fresh_model, X_train, y_train, cv=kf, method='predict')
        meta_features_train[:, i] = oof_pred
    
    # Step 2: Build validation meta-features from validation predictions
    meta_features_val = np.column_stack([r['val_pred'] for r in results])
    
    # Step 3: Get target-specific meta models based on version
    meta_models_safe = get_meta_models_for_target(target_name, len(y_train))
    
    # Display version info
    version_info = {
        'Tg': 'Version 49 (14 meta-models, SVR_poly removed)',
        'FFV': 'Version 48 (15 meta-models, includes SVR_poly)', 
        'Tc': 'Version 52 (14 meta-models, enhanced regularization)',
        'Density': 'Version 52 (14 meta-models, enhanced regularization)',
        'Rg': 'Version 52 (14 meta-models, enhanced regularization)'
    }
    print(f"      Using {version_info.get(target_name, 'Default')} approach")
    
    # Step 4: Evaluate meta-models using proper train/validation split
    best_meta_mae = float('inf')
    best_meta_model = None  
    best_meta_name = None
    best_meta_pred = None
    
    baseline_mae = min([r['val_mae'] for r in results])
    
    print(f"      Testing {len(meta_models_safe)} meta-models...")
    
    for meta_name, meta_model in meta_models_safe.items():
        try:
            # Cross-validate on training meta-features
            cv_scores = -cross_val_score(meta_model, meta_features_train, y_train, 
                                       cv=kf, scoring='neg_mean_absolute_error', n_jobs=1)
            cv_mae = cv_scores.mean()
            
            # Train on training meta-features, predict on validation meta-features
            meta_model.fit(meta_features_train, y_train)
            meta_pred = meta_model.predict(meta_features_val)
            meta_mae = mean_absolute_error(y_val, meta_pred)
            
            print(f"      {meta_name:12}: CV MAE = {cv_mae:.6f} | Val MAE = {meta_mae:.6f}")
            
            # Sanity check: reject unrealistically good scores
            if meta_mae < 0.2 * baseline_mae:
                print(f"      {meta_name:12}: REJECTED (suspiciously low vs baseline {baseline_mae:.6f})")
                continue
            
            # Select best based on validation MAE
            if meta_mae < best_meta_mae:
                best_meta_mae = meta_mae
                best_meta_model = meta_model
                best_meta_name = meta_name
                best_meta_pred = meta_pred
                
        except Exception as e:
            print(f"      {meta_name:12}: Failed ({str(e)[:30]}...)")
            continue
    
    # Fallback to Ridge if all failed
    if best_meta_model is None:
        print(f"      All meta-models failed or rejected, falling back to Ridge...")
        fallback_model = make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE))
        fallback_model.fit(meta_features_train, y_train)
        best_meta_pred = fallback_model.predict(meta_features_val)
        best_meta_mae = mean_absolute_error(y_val, best_meta_pred)
        best_meta_model = fallback_model
        best_meta_name = 'Ridge'
    
    print(f"Best meta-model: {best_meta_name} (MAE: {best_meta_mae:.6f})")
    
    return best_meta_pred, best_meta_model, best_meta_name, best_meta_mae

def apply_clipping(predictions, target_name):
    if target_name in PROPERTY_RANGES and PROPERTY_RANGES[target_name][0] is not None:
        low, high = PROPERTY_RANGES[target_name]
        clipped = np.clip(predictions, low, high)
        n_clipped = np.sum((predictions < low) | (predictions > high))
        if n_clipped > 0:
            print(f"Clipped {n_clipped} out-of-range predictions for {target_name}")
        return clipped
    return predictions

print("Target-specific model training functions ready!")
print("Version-based meta model integration:")
print("   • Tg: Version 49 approach (14 meta-models, optimized)")
print("   • FFV: Version 48 approach (15 meta-models, includes SVR_poly)")
print("   • Tc/Density/Rg: Version 52 approach (14 meta-models, enhanced)")

# Main training loop
print(f"\nTRAINING MODELS FOR ALL TARGETS WITH VERSION-SPECIFIC META LEARNING")
print("=" * 70)

all_target_results = {}
final_models = {}

for target in TARGETS:
    if target in target_splits:
        print(f"\n{'='*50}")
        print(f"TRAINING MODELS FOR {target}")
        print(f"{'='*50}")
        
        # Get data splits
        X_train = target_splits[target]['X_train']
        X_val = target_splits[target]['X_val'] 
        y_train = target_splits[target]['y_train']
        y_val = target_splits[target]['y_val']
        
        # Train base models in parallel
        results = train_target_models_parallel(target, X_train, y_train, X_val, y_val)
        
        # 1. Get best single model
        single_maes = [result['val_mae'] for result in results]
        single_names = [result['name'] for result in results]
        best_single_idx = np.argmin(single_maes)
        best_single_mae = single_maes[best_single_idx]
        best_single_name = single_names[best_single_idx]
        best_single_pred = results[best_single_idx]['val_pred']
        
        # 2. Create weighted ensemble
        weighted_pred, weights = create_weighted_ensemble(results)
        weighted_pred_clipped = apply_clipping(weighted_pred, target)
        weighted_mae = mean_absolute_error(y_val, weighted_pred_clipped)
        
        # 3. Create stacked ensemble
        try:
            stacked_pred, stacked_meta_model = create_stacked_ensemble(results, y_val)
            stacked_pred_clipped = apply_clipping(stacked_pred, target)
            stacked_mae = mean_absolute_error(y_val, stacked_pred_clipped)
        except Exception as e:
            print(f"   Stacking failed for {target}: {e}")
            stacked_pred_clipped = weighted_pred_clipped
            stacked_mae = weighted_mae
            stacked_meta_model = None
        
        # 4. Create VERSION-SPECIFIC meta-learned ensemble
        try:
            meta_pred, meta_model, meta_name, meta_mae_raw = create_meta_learned_ensemble_safe(
                results, X_train, y_train, X_val, y_val, target
            )
            meta_pred_clipped = apply_clipping(meta_pred, target)
            meta_mae = mean_absolute_error(y_val, meta_pred_clipped)
        except Exception as e:
            print(f"   Meta-learning failed for {target}: {e}")
            meta_pred_clipped = weighted_pred_clipped
            meta_mae = weighted_mae
            meta_model = None
            meta_name = "Failed"
        
        # Apply clipping to single model prediction
        best_single_pred_clipped = apply_clipping(best_single_pred, target)
        best_single_mae_clipped = mean_absolute_error(y_val, best_single_pred_clipped)
        
        # FOUR-WAY COMPARISON: Compare all methods
        print(f"Comparing ALL methods for {target}:")
        print(f"   Best Single ({best_single_name}): {best_single_mae_clipped:.6f}")
        print(f"   Weighted Ensemble: {weighted_mae:.6f}")
        print(f"   Stacked Ensemble:  {stacked_mae:.6f}")
        print(f"   Meta-Learned ({meta_name}): {meta_mae:.6f}")
        
        # Select best method among all four
        methods = {
            'single': (best_single_mae_clipped, best_single_pred_clipped, best_single_name, best_single_idx),
            'weighted': (weighted_mae, weighted_pred_clipped, 'Weighted', None),
            'stacked': (stacked_mae, stacked_pred_clipped, 'Stacked', stacked_meta_model),
            'meta': (meta_mae, meta_pred_clipped, f'Meta-{meta_name}', meta_model)
        }
        
        best_method = min(methods.keys(), key=lambda k: methods[k][0])
        best_mae = methods[best_method][0]
        best_pred = methods[best_method][1]
        best_model_info = methods[best_method][2]
        
        print(f"Best method: {best_method.upper()} ({best_model_info}) with MAE {best_mae:.6f}")
        
        # Store results
        all_target_results[target] = {
            'models': results,
            'weights': weights,
            'stacked_meta_model': stacked_meta_model,
            'meta_model': meta_model,
            'meta_name': meta_name,
            'best_method': best_method,
            'best_mae': best_mae,
            'best_single_idx': best_single_idx if best_method == 'single' else None,
            'best_single_name': best_single_name if best_method == 'single' else None
        }

print("\nHYBRID TARGET-SPECIFIC MODEL TRAINING WITH VERSION-BASED META MODELS COMPLETE!")
print("Each target now uses its optimal meta model approach:")
print("   • Tg: Version 49 meta models (streamlined, SVR_poly removed)")
print("   • FFV: Version 48 meta models (comprehensive, includes SVR_poly)")
print("   • Tc/Density/Rg: Version 52 meta models (enhanced regularization)")



print("\nHYBRID TARGET-SPECIFIC MODEL TRAINING START")
print("=" * 70)

# TARGET-SPECIFIC META MODEL CONFIGURATIONS FROM EACH VERSION
def get_meta_models_for_target(target_name, y_train_size):
    """Get target-specific meta models based on version requirements"""
    
    if target_name == 'Tg':
        # Version 49 approach - 14 meta models (SVR_poly removed for efficiency)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }
    
    elif target_name == 'FFV':
        # Version 48 approach - 15 meta models (includes SVR_poly for comprehensive coverage)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=200, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }
    
    else:  # Tc, Density, Rg
        # Version 52 approach - 14 meta models (enhanced regularization, no SVR_poly)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }

def create_meta_learned_ensemble_safe(results, X_train, y_train, X_val, y_val, target_name):
    """Create meta-learned ensemble with target-specific meta models from different versions"""
    print(f"Safe meta-learning for {target_name} using version-specific approach...")
    
    # Step 1: Build training meta-features from OOF predictions on training data ONLY
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    meta_features_train = np.zeros((X_train.shape[0], len(results)))
    
    for i, result in enumerate(results):
        model_func = [func for name, func in get_target_model_config(target_name).items() if name == result['name']][0]
        fresh_model = model_func()
        oof_pred = cross_val_predict(fresh_model, X_train, y_train, cv=kf, method='predict')
        meta_features_train[:, i] = oof_pred
    
    # Step 2: Build validation meta-features from validation predictions
    meta_features_val = np.column_stack([r['val_pred'] for r in results])
    
    # Step 3: Get target-specific meta models based on version
    meta_models_safe = get_meta_models_for_target(target_name, len(y_train))
    
    # Display version info
    version_info = {
        'Tg': 'Version 49 (14 meta-models, SVR_poly removed)',
        'FFV': 'Version 48 (14 meta-models)', 
        'Tc': 'Version 52 (14 meta-models, enhanced regularization)',
        'Density': 'Version 52 (14 meta-models, enhanced regularization)',
        'Rg': 'Version 52 (14 meta-models, enhanced regularization)'
    }
    print(f"      Using {version_info.get(target_name, 'Default')} approach")
    
    # Step 4: Evaluate meta-models using proper train/validation split
    best_meta_mae = float('inf')
    best_meta_model = None  
    best_meta_name = None
    best_meta_pred = None
    
    baseline_mae = min([r['val_mae'] for r in results])
    
    print(f"      Testing {len(meta_models_safe)} meta-models...")
    
    for meta_name, meta_model in meta_models_safe.items():
        try:
            # Cross-validate on training meta-features
            cv_scores = -cross_val_score(meta_model, meta_features_train, y_train, 
                                       cv=kf, scoring='neg_mean_absolute_error', n_jobs=1)
            cv_mae = cv_scores.mean()
            
            # Train on training meta-features, predict on validation meta-features
            meta_model.fit(meta_features_train, y_train)
            meta_pred = meta_model.predict(meta_features_val)
            meta_mae = mean_absolute_error(y_val, meta_pred)
            
            print(f"      {meta_name:12}: CV MAE = {cv_mae:.6f} | Val MAE = {meta_mae:.6f}")
            
            # Sanity check: reject unrealistically good scores
            if meta_mae < 0.2 * baseline_mae:
                print(f"      {meta_name:12}: REJECTED (suspiciously low vs baseline {baseline_mae:.6f})")
                continue
            
            # Select best based on validation MAE
            if meta_mae < best_meta_mae:
                best_meta_mae = meta_mae
                best_meta_model = meta_model
                best_meta_name = meta_name
                best_meta_pred = meta_pred
                
        except Exception as e:
            print(f"      {meta_name:12}: Failed ({str(e)[:30]}...)")
            continue
    
    # Fallback to Ridge if all failed
    if best_meta_model is None:
        print(f"      All meta-models failed or rejected, falling back to Ridge...")
        fallback_model = make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE))
        fallback_model.fit(meta_features_train, y_train)
        best_meta_pred = fallback_model.predict(meta_features_val)
        best_meta_mae = mean_absolute_error(y_val, best_meta_pred)
        best_meta_model = fallback_model
        best_meta_name = 'Ridge'
    
    print(f"   Best meta-model: {best_meta_name} (MAE: {best_meta_mae:.6f})")
    
    return best_meta_pred, best_meta_model, best_meta_name, best_meta_mae

all_target_results = {}
final_models = {}

for target in TARGETS:
    if target in target_splits:
        print(f"{'='*50}\nTRAINING HYBRID MODELS FOR {target}\n{'='*50}")
        X_train = target_splits[target]['X_train']
        X_val   = target_splits[target]['X_val']
        y_train = target_splits[target]['y_train']
        y_val   = target_splits[target]['y_val']
        
        results = train_target_models_parallel(target, X_train, y_train, X_val, y_val)
        
        # 1. Get best single model
        single_maes = [result['val_mae'] for result in results]
        single_names = [result['name'] for result in results]
        best_single_idx = np.argmin(single_maes)
        best_single_mae = single_maes[best_single_idx]
        best_single_name = single_names[best_single_idx]
        best_single_pred = results[best_single_idx]['val_pred']
        
        # 2. Create weighted ensemble
        weighted_pred, weights = create_weighted_ensemble(results)
        weighted_pred_clipped = apply_clipping(weighted_pred, target)
        weighted_mae = mean_absolute_error(y_val, weighted_pred_clipped)
        
        # 3. Create stacked ensemble
        try:
            stacked_pred, stacked_meta_model = create_stacked_ensemble(results, y_val)
            stacked_pred_clipped = apply_clipping(stacked_pred, target)
            stacked_mae = mean_absolute_error(y_val, stacked_pred_clipped)
        except Exception as e:
            print(f"   Stacking failed for {target}: {e}")
            stacked_pred_clipped = weighted_pred_clipped
            stacked_mae = weighted_mae
            stacked_meta_model = None
        
        # 4. Create VERSION-SPECIFIC meta-learned ensemble
        try:
            meta_pred, meta_model, meta_name, meta_mae_raw = create_meta_learned_ensemble_safe(
                results, X_train, y_train, X_val, y_val, target
            )
            meta_pred_clipped = apply_clipping(meta_pred, target)
            meta_mae = mean_absolute_error(y_val, meta_pred_clipped)
        except Exception as e:
            print(f"   Meta-learning failed for {target}: {e}")
            meta_pred_clipped = weighted_pred_clipped
            meta_mae = weighted_mae
            meta_model = None
            meta_name = "Failed"
        
        # Apply clipping to single model prediction
        best_single_pred_clipped = apply_clipping(best_single_pred, target)
        best_single_mae_clipped = mean_absolute_error(y_val, best_single_pred_clipped)
        
        # FOUR-WAY COMPARISON: Compare all methods including meta-learning
        print(f"Comparing ALL methods for {target}:")
        print(f"Best Single ({best_single_name}): {best_single_mae_clipped:.6f}")
        print(f"Weighted Ensemble: {weighted_mae:.6f}")
        print(f"Stacked Ensemble:  {stacked_mae:.6f}")
        print(f"Meta-Learned ({meta_name}): {meta_mae:.6f}")
        
        # Select best method among all four
        methods = {
            'single': (best_single_mae_clipped, best_single_pred_clipped, best_single_name, best_single_idx),
            'weighted': (weighted_mae, weighted_pred_clipped, 'Weighted', None),
            'stacked': (stacked_mae, stacked_pred_clipped, 'Stacked', stacked_meta_model),
            'meta': (meta_mae, meta_pred_clipped, f'Meta-{meta_name}', meta_model)
        }
        
        best_method = min(methods.keys(), key=lambda k: methods[k][0])
        best_mae = methods[best_method][0]
        best_pred = methods[best_method][1]
        best_model_info = methods[best_method][2]
        
        print(f"Best method: {best_method.upper()} ({best_model_info}) with MAE {best_mae:.6f}")
        
        all_target_results[target] = {
            'models': results,
            'weights': weights,
            'stacked_meta_model': stacked_meta_model,
            'meta_model': meta_model,
            'meta_name': meta_name,
            'best_method': best_method,
            'best_mae': best_mae,
            'best_single_idx': best_single_idx if best_method == 'single' else None,
            'best_single_name': best_single_name if best_method == 'single' else None
        }
        
        print(f"Retraining {best_method} on full dataset with target-specific optimization...")
        X_full = np.vstack([X_train, X_val])
        y_full = np.concatenate([y_train, y_val])
        
        if best_method == 'single':
            target_models_config = get_target_model_config(target)
            best_model_func = list(target_models_config.values())[best_single_idx]
            final_model = best_model_func()
            final_model.fit(X_full, y_full)
            
            final_models[target] = {
                'method': 'single',
                'model': final_model,
                'model_name': best_single_name
            }
        elif best_method == 'meta':
            # Retrain base models + meta-model on full data
            target_models_config = get_target_model_config(target)
            final_base_models = []
            for model_func in target_models_config.values():
                model = model_func()
                model.fit(X_full, y_full)
                final_base_models.append(model)
            
            # Get OOF predictions for meta-model training
            oof_preds = []
            for model in final_base_models:
                oof_pred = cross_val_predict(model, X_full, y_full, cv=5, method='predict')
                oof_preds.append(oof_pred)
            
            X_meta_full = np.column_stack(oof_preds)
            
            # Create fresh meta-model and train on full OOF predictions
            if meta_name in ['RandomForest', 'ExtraTrees', 'LightGBM_meta', 'CatBoost_meta', 'XGBoost_meta']:
                meta_model_funcs = {
                    'RandomForest': lambda: RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
                    'ExtraTrees': lambda: ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
                    'LightGBM_meta': lambda: LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
                    'CatBoost_meta': lambda: CatBoostRegressor(iterations=200, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
                    'XGBoost_meta': lambda: XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0)
                }
                final_meta_model = meta_model_funcs.get(meta_name, lambda: RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE))()
            else:
                meta_model_funcs = {
                    'Ridge': lambda: make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
                    'ElasticNet': lambda: make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
                    'Lasso': lambda: make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
                    'BayesianRidge': lambda: make_pipeline(RobustScaler(), BayesianRidge()),
                    'PLS': lambda: make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
                    'SVR_linear': lambda: make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
                    'SVR_rbf': lambda: make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
                    'SVR_poly': lambda: make_pipeline(RobustScaler(), SVR(kernel='poly', degree=2, C=0.5, epsilon=0.02)),  # Only for FFV
                    'KNN': lambda: make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, len(y_full)//20), weights='distance')),
                    'MLP_meta': lambda: create_target_specific_mlp(target, RANDOM_STATE)
                }
                final_meta_model = meta_model_funcs.get(meta_name, lambda: make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)))()
            
            final_meta_model.fit(X_meta_full, y_full)
            
            final_models[target] = {
                'method': 'meta',
                'models': final_base_models,
                'meta_model': final_meta_model,
                'meta_name': meta_name
            }
        else:
            target_models_config = get_target_model_config(target)
            final_target_models = []
            for model_func in target_models_config.values():
                model = model_func()
                model.fit(X_full, y_full)
                final_target_models.append(model)
            
            final_models[target] = {
                'models': final_target_models,
                'weights': weights,
                'stacked_meta_model': stacked_meta_model,
                'method': best_method
            }

print("\nHYBRID TARGET-SPECIFIC MODEL TRAINING WITH VERSION-BASED META MODELS COMPLETE!")
print("Each target now uses its optimal meta model approach:")
print("   • Tg: Version 49 meta models (14 models, streamlined)")
print("   • FFV: Version 48 meta models (15 models, includes SVR_poly)")
print("   • Tc/Density/Rg: Version 52 meta models (14 models, enhanced regularization)")
print("Model training complete!")

# TARGET-SPECIFIC META MODEL CONFIGURATIONS FROM EACH VERSION
def get_meta_models_for_target(target_name, y_train_size):
    """Get target-specific meta models based on version requirements"""
    
    if target_name == 'Tg':
        # Version 49 approach - 14 meta models (SVR_poly removed for efficiency)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }
    
    elif target_name == 'FFV':
        # Version 48 approach - 15 meta models (includes SVR_poly for comprehensive coverage)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            'SVR_poly': make_pipeline(RobustScaler(), SVR(kernel='poly', degree=2, C=0.5, epsilon=0.02)),  # ✅ KEPT for FFV (Version 48)
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }
    
    else:  # Tc, Density, Rg
        # Version 52 approach - 14 meta models (enhanced regularization, no SVR_poly)
        return {
            'Ridge': make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE)),
            'ElasticNet': make_pipeline(RobustScaler(), ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)),
            'Lasso': make_pipeline(RobustScaler(), Lasso(alpha=0.01, max_iter=5000, random_state=RANDOM_STATE)),
            'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge()),
            'PLS': make_pipeline(RobustScaler(), PLSRegression(n_components=min(2, 3))),
            'SVR_linear': make_pipeline(RobustScaler(), SVR(kernel='linear', C=1.0, epsilon=0.01)),
            'SVR_rbf': make_pipeline(RobustScaler(), SVR(kernel='rbf', C=1.0, epsilon=0.02)),
            # SVR_poly REMOVED - Version 52 optimization
            'KNN': make_pipeline(RobustScaler(), KNeighborsRegressor(n_neighbors=max(5, y_train_size//20), weights='distance')),
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
            'ExtraTrees': ExtraTreesRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE),
            'LightGBM_meta': LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_samples=10, random_state=RANDOM_STATE, verbosity=-1),
            'CatBoost_meta': CatBoostRegressor(iterations=100, learning_rate=0.05, depth=3, min_data_in_leaf=10, random_seed=RANDOM_STATE, verbose=False),
            'XGBoost_meta': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=10, random_state=RANDOM_STATE, verbosity=0),
            'MLP_meta': create_target_specific_mlp(target_name, RANDOM_STATE)
        }

def create_meta_learned_ensemble_safe(results, X_train, y_train, X_val, y_val, target_name):
    """Create meta-learned ensemble with target-specific meta models from different versions"""
    print(f"   🧠 Safe meta-learning for {target_name} using version-specific approach...")
    
    # Step 1: Build training meta-features from OOF predictions on training data ONLY
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    meta_features_train = np.zeros((X_train.shape[0], len(results)))
    
    for i, result in enumerate(results):
        model_func = [func for name, func in get_target_model_config(target_name).items() if name == result['name']][0]
        fresh_model = model_func()
        oof_pred = cross_val_predict(fresh_model, X_train, y_train, cv=kf, method='predict')
        meta_features_train[:, i] = oof_pred
    
    # Step 2: Build validation meta-features from validation predictions
    meta_features_val = np.column_stack([r['val_pred'] for r in results])
    
    # Step 3: Get target-specific meta models based on version
    meta_models_safe = get_meta_models_for_target(target_name, len(y_train))
    
    # Display version info
    version_info = {
        'Tg': 'Version 49 (14 meta-models, SVR_poly removed)',
        'FFV': 'Version 48 (15 meta-models, includes SVR_poly)', 
        'Tc': 'Version 52 (14 meta-models, enhanced regularization)',
        'Density': 'Version 52 (14 meta-models, enhanced regularization)',
        'Rg': 'Version 52 (14 meta-models, enhanced regularization)'
    }
    print(f"      Using {version_info.get(target_name, 'Default')} approach")
    
    # Step 4: Evaluate meta-models using proper train/validation split
    best_meta_mae = float('inf')
    best_meta_model = None  
    best_meta_name = None
    best_meta_pred = None
    
    baseline_mae = min([r['val_mae'] for r in results])
    
    print(f"      Testing {len(meta_models_safe)} meta-models...")
    
    for meta_name, meta_model in meta_models_safe.items():
        try:
            # Cross-validate on training meta-features
            cv_scores = -cross_val_score(meta_model, meta_features_train, y_train, 
                                       cv=kf, scoring='neg_mean_absolute_error', n_jobs=1)
            cv_mae = cv_scores.mean()
            
            # Train on training meta-features, predict on validation meta-features
            meta_model.fit(meta_features_train, y_train)
            meta_pred = meta_model.predict(meta_features_val)
            meta_mae = mean_absolute_error(y_val, meta_pred)
            
            print(f"      {meta_name:12}: CV MAE = {cv_mae:.6f} | Val MAE = {meta_mae:.6f}")
            
            # Sanity check: reject unrealistically good scores
            if meta_mae < 0.2 * baseline_mae:
                print(f"      {meta_name:12}: REJECTED (suspiciously low vs baseline {baseline_mae:.6f})")
                continue
            
            # Select best based on validation MAE
            if meta_mae < best_meta_mae:
                best_meta_mae = meta_mae
                best_meta_model = meta_model
                best_meta_name = meta_name
                best_meta_pred = meta_pred
                
        except Exception as e:
            print(f"      {meta_name:12}: Failed ({str(e)[:30]}...)")
            continue
    
    # Fallback to Ridge if all failed
    if best_meta_model is None:
        print(f"      All meta-models failed or rejected, falling back to Ridge...")
        fallback_model = make_pipeline(RobustScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE))
        fallback_model.fit(meta_features_train, y_train)
        best_meta_pred = fallback_model.predict(meta_features_val)
        best_meta_mae = mean_absolute_error(y_val, best_meta_pred)
        best_meta_model = fallback_model
        best_meta_name = 'Ridge'
    
    print(f"Best meta-model: {best_meta_name} (MAE: {best_meta_mae:.6f})")
    
    return best_meta_pred, best_meta_model, best_meta_name, best_meta_mae

print("GENERATING FINAL HYBRID TEST PREDICTIONS\n" + "=" * 50)
test_predictions = {}
for target in TARGETS:
    if target in final_models:
        print(f"Predicting {target} with hybrid optimization...")
        target_info = final_models[target]
        method = target_info['method']
        X_test_target = target_test_features[target]
        
        if method == 'single':
            pred = target_info['model'].predict(X_test_target)
            print(f"   Method: Hybrid Single ({target_info['model_name']})")
            
        elif method == 'weighted':
            pred = np.zeros(X_test_target.shape[0])
            for model, weight in zip(target_info['models'], target_info['weights']):
                pred += weight * model.predict(X_test_target)
            print(f"   Method: Hybrid Weighted Ensemble")
            
        elif method == 'stacked':
            model_preds = np.column_stack([model.predict(X_test_target) for model in target_info['models']])
            pred = target_info['stacked_meta_model'].predict(model_preds)
            print(f"   Method: Hybrid Stacked Ensemble")
            
        elif method == 'meta':
            #: Meta-learned prediction with version-specific meta models
            base_preds = np.column_stack([
                model.predict(X_test_target) for model in target_info['models']
            ])
            pred = target_info['meta_model'].predict(base_preds)
            version_map = {
                'Tg': 'Version 49',
                'FFV': 'Version 48', 
                'Tc': 'Version 52',
                'Density': 'Version 52',
                'Rg': 'Version 52'
            }
            version_used = version_map.get(target, 'Default')
            print(f"   Method: Hybrid Meta-Learned ({target_info['meta_name']}) - {version_used}")
        
        pred_clipped = apply_clipping(pred, target)
        pred_clipped = np.asarray(pred_clipped).flatten()
        test_predictions[target] = pred_clipped
        
        print(f"   Range: [{pred_clipped.min():.3f}, {pred_clipped.max():.3f}]")

print("CREATING HYBRID SUBMISSION")
print("=" * 30)
submission_data = {'id': test_df['id']}
for target in TARGETS:
    if target in test_predictions:
        submission_data[target] = test_predictions[target]
    else:
        submission_data[target] = np.zeros(len(test_df))

expected_len = len(test_df['id'])
for key, value in submission_data.items():
    arr = np.asarray(value).flatten()
    if arr.ndim > 1:
        arr = arr.flatten()
    if len(arr) != expected_len:
        arr = np.resize(arr, expected_len)
    submission_data[key] = arr.astype(np.float64)

submission = pd.DataFrame(submission_data)
submission.to_csv('submission.csv', index=False)
print(f"Hybrid submission saved: submission.csv\nShape: {submission.shape}")

# Performance summary with method tracking
print(f"\nFINAL HYBRID PERFORMANCE SUMMARY")
print("=" * 70)

validation_maes = []
method_counts = {'single': 0, 'weighted': 0, 'stacked': 0, 'meta': 0}  # Added 'meta'
version_usage = {'Version 48': [], 'Version 49': [], 'Version 52': []}

for target in TARGETS:
    if target in all_target_results:
        res = all_target_results[target]
        mae = res.get('best_mae')
        method = res.get('best_method')
        
        # Track version usage
        if target == 'Tg':
            version_usage['Version 49'].append(target)
        elif target == 'FFV':
            version_usage['Version 48'].append(target)
        else:
            version_usage['Version 52'].append(target)
        
        if method == 'single':
            label = res.get('best_single_name', 'Single')
        elif method == 'meta':  # Added meta method handling
            meta_name = res.get('meta_name', 'Unknown')
            label = f"Meta-{meta_name}"
        else:
            label = method.capitalize() if method else "Unknown"
        
        if mae is not None and method is not None:
            validation_maes.append(mae)
            method_counts[method] = method_counts.get(method, 0) + 1
            print(f"{target:8}: {label:15} | MAE: {mae:.6f}")
        else:
            print(f"{target:8}: {label:15} | MAE: N/A")

# Version usage summary
print(f"\n📊 Hybrid Version Usage Summary:")
for version, targets in version_usage.items():
    if targets:
        meta_info = ""
        if version == "Version 49":
            meta_info = " (14 meta-models, optimized)"
        elif version == "Version 48":
            meta_info = " (15 meta-models, comprehensive)"
        elif version == "Version 52":
            meta_info = " (14 meta-models, enhanced)"
        print(f"   {version:12}: {', '.join(targets)}{meta_info}")

# Method selection summary  
print(f"\nHybrid Method Selection Summary:")
for method, count in method_counts.items():
    if count > 0:
        method_description = {
            'single': f"{method.capitalize()} (Best individual model)",
            'weighted': f"{method.capitalize()} (Inverse MAE weighting)",
            'stacked': f"{method.capitalize()} (Ridge meta-learner)",
            'meta': f"{method.capitalize()} (Version-specific meta-learning)"
        }
        print(f"   {method_description.get(method, method.capitalize()):35}: {count} targets")

# Calculate average with safe handling
average_mae = np.mean(validation_maes) if validation_maes else float('nan')
avg_str = f"{average_mae:.6f}" if not np.isnan(average_mae) else "N/A"
print(f"\nAverage Hybrid Validation MAE: {avg_str}")

if not np.isnan(average_mae):
    if average_mae < 0.065:
        improvement = 0.068 - average_mae  # Compare against user's 0.068 baseline
        print(f"OUTSTANDING! Improvement vs 0.068 baseline: -{improvement:.6f}")
        print(f"You've broken into elite territory!")
    elif average_mae < 0.068:
        improvement = 0.068 - average_mae
        print(f"EXCELLENT! Improvement vs 0.068 baseline: -{improvement:.6f}")
        print(f"You've beaten your best model!")
    elif average_mae < 0.070:
        improvement = 0.070 - average_mae
        print(f"GOOD! Close to your 0.068 target: {average_mae:.6f}")
        print(f"Consider CatBoost-only architecture next")
    else:
        print(f"Results: {average_mae:.6f} - Time to pivot to new architecture")
        print(f"Recommend trying CatBoost-only approach")

print(f"\nHYBRID TARGET-SPECIFIC MODEL SELECTION WITH META-LEARNING COMPLETE!")
print("KEY FEATURES:")
print("   • Target-Specific Optimization: Each target uses its best-performing version")
print("   • Tg: Version 49 configuration (14 meta-models, optimized)")
print("   • FFV: Version 48 configuration (15 meta-models, comprehensive)")  
print("   • Tc/Density/Rg: Version 52 configuration (14 meta-models, enhanced)")
print("   • Four-Way Method Selection: Single, Weighted, Stacked, Meta-Learning")
print("   • Version-Specific Meta Models: Each target uses its proven best meta approach")
print("   • Data-driven Clipping: Safe prediction bounds based on actual data")

print(f"\nExpected Results:")
print(f"   • This hybrid approach combines the best of your previous versions")
print(f"   • Each target gets its proven best feature extraction approach")
print(f"   • Meta-learning uses version-specific optimizations")
print(f"   • Four methods compete, winner selected automatically")
print(f"   • You should see results competitive with or better than your 0.068 baseline")

print("\nENHANCED HYBRID PIPELINE COMPLETE - CHECK YOUR RESULTS!")
